# मानव-इन-द-लूप: प्रि-एक्शन गेटहरू, जोखिम स्तर निर्धारण, र अडिट लगिङ

यस पाठको README ले मानव-इन-द-लूपलाई छोटो स्निपेटसँग परिचय गराउँछ जसले एजेन्टले जवाफ उत्पादन गरेपछि प्रयोगकर्तालाई `APPROVE` वा `REJECT` सोध्छ। त्यो ढाँचा सुरुवातका लागि राम्रो हो, तर उत्पादन HITL कार्यान्वयनहरू प्रायः तीन थप भागहरू आवश्यक हुन्छन्:

1. एक **प्रि-एक्शन गेट** जुन एजेन्टले जोखिमपूर्ण कदम चाल्नुअघि नै चल्छ, ताकि लागत, अपरिवर्तनीयता, र विलम्बतालाई नियन्त्रणमा राख्न सकियोस्।
2. **जोखिम स्तर निर्धारण**, जसले कम जोखिमका कार्यहरू स्वचालित रूपमा सञ्चालन गर्छ, मध्यम जोखिमका कार्यहरू ब्याच-अप्रूभ हुन्छन्, र केवल उच्च जोखिमका कार्यहरूमा मान्छेले अवरोध गर्छ।
3. एक **अडिट लग प्लस संशोधन लूप**, जसले प्रत्येक गेट निर्णयलाई JSONL मा राख्छ, र अस्वीकृत गर्दा एजेन्टलाई `Revising...` मात्र प्रिन्ट गर्नुभन्दा संरचित कारणसहित पुन:प्रेरित गर्छ।

यो नोटबुकले यी सबैलाई `06-system-message-framework.ipynb` का समान प्रमिकहरूमा आधारित बनाएर निर्माण गर्दछ। यो `DEMO_MODE = True` मा अन्त्यदेखि अन्त्यसम्म चल्छ (कुनै पनि इन्टरएक्टिभ इनपुट आवश्यक छैन) वा `DEMO_MODE = False` हुँदा वास्तविक `input()` प्राम्प्टहरू सहित चल्न सक्छ। ध्यान दिनुहोस्: DEMO_MODE मा तेस्रो लक्ष्यको पुनः प्रयास स्क्रिप्ट गरिएको हुन्छ जसले लूपको मेकानिक्सलाई अन्त्यदेखि अन्त्यसम्म देखाउँछ। वास्तविक पुनरावृत्ति-चालित पुनः वर्गीकरणका लागि `DEMO_MODE = False` र एक अपरेटर आवश्यक पर्छ।

**परिधि बाहिर (अन्य पाठहरूमा सम्हालिएको):** प्रमाणीकरण र पहुँच नियन्त्रण (पाठ 06 README खतरा 2), उपकरण-काल मिडलवेयर (पाठ 14 MAF गहिरो विश्लेषण), बहु-एजेन्ट बहस ढाँचाहरू।

In [ ]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

load_dotenv()

DEMO_MODE = True  # set False to use real input() prompts

# Per-run unique log filename so demo runs don't overwrite each other and
# the notebook doesn't touch any pre-existing gate_log.jsonl in the working
# directory.
GATE_LOG_PATH = Path(
    f"gate_log_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.jsonl"
)

token = os.environ.get("GITHUB_TOKEN", "")
if not token:
    raise RuntimeError(
        "GITHUB_TOKEN environment variable is not set. This notebook needs "
        "a GitHub PAT with 'Models: Read-only' permission. Set GITHUB_TOKEN "
        "in your environment or a local .env file before running."
    )

endpoint = "https://models.github.ai/inference"
model_name = "gpt-4o"

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


## Pattern 1: पूर्व-कार्य द्वार

README को HITL स्निपेटले एजेन्टलाई पहिलो कल गर्छ, त्यसपछि प्रयोगकर्तालाई आउटपुट अनुमोदन गर्न सोध्छ। त्यो एक **पश्च-कार्य** प्रवाह हो। एजेन्टले पहिले नै कार्य सम्पन्न गरिसकेको हुन्छ, त्यसैले LLM कल लागत पहिले नै तिरी सकिएकै हुन्छ, र कुनै पनि पार्श्व प्रभाव (पठाइएको इमेल, लेखिएको डाटाबेस पङ्क्ति, पोस्ट गरिएको टिप्पणी) पहिले नै भइसकेको हुन्छ।

एक **पूर्व-कार्य** प्रवाहले जोखिमपूर्ण चरण सुरु हुनु अघि द्वार राख्छ। एजेन्टले कार्य प्रस्ताव गर्छ, द्वारले कार्यान्वयन गर्ने कि नगर्ने निर्णय गर्छ, र केवल अनुमोदनमा मात्र पार्श्व प्रभाव हुन्छ।

| पक्ष | पश्च-कार्य अनुमोदन (README स्निपेट) | पूर्व-कार्य द्वार (यो नोटबुक) |
|---|---|---|
| अनुमोदन कहिले चल्छ? | एजेन्टले आउटपुट बनाइसकेपछि | कुनै पनि पार्श्व-प्रभाव कार्यान्वयन हुनु अघि |
| अस्वीकृतिमा LLM लागत | पहिले नै तिर्ई सकेको | केवल प्रस्तावको लागि तिर्ने, कार्यको लागि होइन |
| अपरिवर्तनीय पार्श्व प्रभाव | सम्भव (कार्य पहिले नै भइसकेको) | रोकिएको |
| लेखापरीक्षण स्पष्टता | अनुमोदन एक प्रिन्ट वक्तव्य हो | अनुमोदन एक JSON रेकर्ड हो जसमा टाइमस्ट्याम्प, कार्य, कारण हुन्छ |


In [ ]:
def gate_action(action_description: str, risk_tier: str, attempt: int = 0) -> dict:
    """Run a single pre-action gate.

    Returns a decision dict with keys: decision, reason, ts.
    Decision is one of: approve, deny, escalate.
    Safe default on EOF or unexpected input is deny.

    DEMO_MODE behavior: high-risk actions are denied on attempt 0 and
    auto-approved on attempt >= 1. This is scripted approval to show the
    loop mechanics (deny -> retry -> approve). It is NOT revision-driven
    re-classification. Real revision-driven re-classification requires
    DEMO_MODE=False and a human operator who evaluates the revised
    proposal on its own merits.
    """
    print(f"[gate] proposed action ({risk_tier}, attempt={attempt}): {action_description}")

    if DEMO_MODE:
        if risk_tier == "high":
            decision = "approve" if attempt >= 1 else "deny"
            reason = (
                "DEMO_MODE: scripted approval on retry to show loop mechanics"
                if attempt >= 1
                else "DEMO_MODE: high risk denied on first attempt"
            )
        else:
            decision = "approve"
            reason = f"DEMO_MODE canned response for tier={risk_tier}"
    else:
        try:
            raw = input("[gate] approve / deny / escalate? ").strip().lower()
        except EOFError:
            raw = ""
        if raw in {"approve", "deny", "escalate"}:
            decision, reason = raw, "operator input"
        elif raw == "":
            decision, reason = "deny", "no input received, defaulted to deny"
        else:
            decision, reason = "deny", f"invalid input {raw!r}, defaulted to deny"

    return {
        "decision": decision,
        "reason": reason,
        "action": action_description,
        "risk_tier": risk_tier,
        "ts": datetime.now(timezone.utc).isoformat(),
    }


## ढाँचा २: जोखिम चरणबद्धी

हरेक क्रियाकलापलाई मानवीय अनुमोदनको आवश्यकता हुँदैन। सार्वजनिक API सँगको केवल पढ्न सकिने लुकअपले ग्राहकलाई इमेल पठाउने भन्दा फरक जोखिम हुन्छ। दुवैलाई समान रुपमा व्यवहार गर्दा अपरेटरको ध्यान व्यर्थ हुन्छ र एजेन्टको गति सुस्तिन्छ।

सजिलो ३-चरण मोडल:

| चरण | उदाहरणहरू | अनुमोदन प्रक्रिया |
|---|---|---|
| `low` (पढ्न मात्र) | ज्ञान आधार खोज्नु, उडान विकल्पहरू खोज्नु, सार्वजनिक वेब पृष्ठ लिनु | स्वचालित रूपमा कार्यान्वयन, अडिटको लागि लग गरिएको |
| `medium` (सस्तो परिवर्तन) | नतिजा क्यास गर्नु, काउन्टर बढाउनु, रिमाइन्डर तालिका गर्नु | स्वचालित रूपमा कार्यान्वयन, तर दैनिक समीक्षा सामूहिक रूपमा |
| `high` (बाह्य-सामना वा अपरिवर्तनीय) | इमेल पठाउनु, कार्ड चार्ज गर्नु, सार्वजनिक च्यानलमा पोष्ट गर्नु | मानवीय अनुमोदनमा रोक्नु |

यो एउटा चरणबद्धी हो। उत्पादन प्रणालीहरूले प्रायः धेरै सूक्ष्म चरणहरू प्रयोग गर्छन् (जस्तै, AWS IAM अनुमति स्तरहरू, भूमिका-आधारित पहुँच चरणहरू)। तलको ३-चरण संस्करण पढ्न मात्र र साइड-इफेक्ट गर्ने कार्यहरू मिश्रण गर्ने एजेन्टका लागि सबैभन्दा सानो उपयोगी संस्करण हो।

तलको वर्गीकरणकर्ता शब्दावली नियमहरू प्रयोग गर्दछ ताकि डेमो निश्चित र सस्तो रहोस्। उत्पादन प्रणालीमा तपाईं यसलाई सिकाइएका वर्गीकरणकर्ता वा नीति इञ्जिनसँग बदल्नुहुनेछ।

In [ ]:
LOW_RISK_KEYWORDS = {
    "look", "lookup", "search", "fetch", "read", "query", "view",
    "get", "list", "weather", "summarize",
}
HIGH_RISK_KEYWORDS = {
    "send", "email", "post", "publish", "charge", "pay", "transfer",
    "delete", "drop", "cancel", "refund",
}
MEDIUM_RISK_KEYWORDS = {
    "cache", "schedule", "reminder", "book", "reserve", "update",
    "increment", "log",
}

AUTO_APPROVE_REASONS = {
    "low": "auto-approved (low risk)",
    "medium": "auto-approved (medium risk, queued for batched review)",
}


def classify_risk(action: str) -> str:
    """Classify an action string into one of: low, medium, high.

    Keyword-based heuristic. Checks high-risk first (most severe), then
    low-risk explicit reads, then medium-risk mutations. Unrecognized
    actions default to medium, not low.

    Default for unrecognized actions is 'medium', not 'low'. A read-only
    keyword set will always have blind spots, and the parent README's
    threat list (critical-system access, knowledge-base poisoning,
    cascading errors) all involve cases an action-name alone cannot rule
    out. Routing unknown actions through batched review is the safer
    default than auto-executing them.
    """
    text = action.lower()
    if any(kw in text for kw in HIGH_RISK_KEYWORDS):
        return "high"
    if any(kw in text for kw in LOW_RISK_KEYWORDS):
        return "low"
    if any(kw in text for kw in MEDIUM_RISK_KEYWORDS):
        return "medium"
    # Explicit fail-safe default: unrecognized actions route to batched review.
    return "medium"


def tiered_gate(action: str, attempt: int = 0) -> dict:
    """Classify then gate. Low and medium tiers auto-approve; high blocks."""
    tier = classify_risk(action)
    if tier in AUTO_APPROVE_REASONS:
        return {
            "decision": "approve",
            "reason": AUTO_APPROVE_REASONS[tier],
            "action": action,
            "risk_tier": tier,
            "ts": datetime.now(timezone.utc).isoformat(),
        }
    return gate_action(action, tier, attempt=attempt)


## ढाँचा ३: अडिट लग र संशोधन लूप

`print("Response approved.")` एउटा अडिट लग होइन। विश्वासका लागि, प्रत्येक गेट निर्णयलाई संरचित घटनाका रूपमा रेकर्ड गर्नुपर्छ जुन तपाईं पछि सोध्न, पुन:चलाउन, वा घटनाको समीक्षा सँग जोड्न सक्नुहुन्छ।

दुई भागहरू:

1. **Append-only JSONL।** निर्णय अनुसार एक लाइन, जसमा टाइमस्ट्याम्प, कार्य, तह, निर्णय, कारण हुन्छ। सजिलै grep गर्न मिल्छ, पछि वास्तविक लग स्टोरमा पठाउन सजिलो।
2. **अस्वीकृतिमा संशोधन लूप।** जब गेटले `deny` फर्काउँछ, एजेण्टले आफूलाई अस्वीकृति कारणसँगै पुनःप्रश्न गर्छ, ताकि अर्को प्रस्ताव समस्याबाट बच्न सकोस्।

In [ ]:
def log_decision(decision: dict) -> None:
    """Append a gate decision to the JSONL audit log."""
    with GATE_LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(decision) + "\n")


def propose_action(goal: str, prior_rejection: str | None = None) -> str:
    """Ask the LLM to propose a concrete next action for a goal.

    If prior_rejection is provided, it is fed back so the LLM can avoid
    the same failure mode in the next proposal.
    """
    system = (
        "You are an action planner for an agent. Propose ONE concrete next\n"
        "action (a single sentence) toward the user's goal. If a prior\n"
        "rejection reason is given, propose a different action that addresses\n"
        "the rejection."
    )
    user_text = f"Goal: {goal}"
    if prior_rejection:
        user_text += f"\n\nPrior proposal was denied. Reason: {prior_rejection}"

    response = client.complete(
        model=model_name,
        messages=[SystemMessage(content=system), UserMessage(content=user_text)],
    )
    return response.choices[0].message.content.strip()


def run_with_revision(goal: str, max_revisions: int = 2) -> dict:
    """Propose, gate, and on rejection revise up to max_revisions times."""
    prior_reason: str | None = None
    for attempt in range(max_revisions + 1):
        action = propose_action(goal, prior_rejection=prior_reason)
        decision = tiered_gate(action, attempt=attempt)
        decision["attempt"] = attempt
        log_decision(decision)
        if decision["decision"] == "approve":
            return decision
        prior_reason = decision["reason"]
    return {**decision, "final": "max_revisions_reached"}


In [ ]:
# End-to-end demo: three goals at three different risk profiles.
# GATE_LOG_PATH is per-run (timestamped) so no prior log is touched.

goals = [
    "Look up the weather in Seattle for the customer's trip planning.",
    "Schedule a reminder for the customer to check in 24 hours before their flight.",
    "Send a marketing email to the customer about premium upgrade options.",
]

for goal in goals:
    print(f"\n=== Goal: {goal} ===")
    outcome = run_with_revision(goal, max_revisions=1)
    print(f"[final] {outcome['decision']} ({outcome['reason']})")

print(f"\n=== Audit log ({GATE_LOG_PATH.name}) ===")
for line in GATE_LOG_PATH.read_text(encoding="utf-8").splitlines():
    record = json.loads(line)
    print(f"  [{record['risk_tier']:6s}] {record['decision']:8s} "
          f"attempt={record.get('attempt', '?')} action={record['action'][:140]}")


## थप स्रोतहरू

केही अन्य सार्वजनिक परियोजनाहरू यी HITL ढाँचाहरूका भिन्नता कार्यान्वयन गर्दछन्। तपाईंको स्ट्याकमा के उपयुक्त हुन्छ भनी पत्ता लगाउन तिनीहरूको दृष्टिकोणहरू तुलना गर्नुहोस्:

- **LangChain** human-in-the-loop उपकरण रैपरहरू ([डक्स](https://python.langchain.com/docs/integrations/tools/human_tools)): मानव इनपुटको लागि कार्यान्वयन रोक्ने ड्रप-इन उपकरण रैपरहरू।
- **AutoGen** `UserProxyAgent` ([v0.2 डक्स](https://microsoft.github.io/autogen/0.2/docs/topics/human-in-the-loop); AutoGen v0.4+ ले यसलाई पुनर्संरचना गरेको छ): बहु-एजेन्ट संवादहरूमा मानवलाई प्रतिनिधित्व गर्न विशेष रूपमा एजेन्ट भूमिका प्रयोग गर्दछ।
- **Semantic Kernel** फङ्क्शन फिल्टर्स ([डक्स](https://learn.microsoft.com/en-us/semantic-kernel/concepts/enterprise-readiness/filters)): प्रत्येक फङ्क्शन कल वरिपरि चल्ने मिडलवेयर, गेटिङ लॉजिकका लागि उपयुक्त।

हरेक परियोजनाले ती तीन उप-ढाँचाहरूलाई फरक तरिकाले ह्यान्डल गर्दछ: LangChain ले तिनलाई उपकरणको रूपमा रैपर गर्दछ, AutoGen ले एजेन्ट भूमिका प्रयोग गर्दछ, Semantic Kernel ले मिडलवेयर फिल्टर प्रयोग गर्दछ। आफ्नो एजेन्टका लागि डिजाइन छनोट गर्नु अघि एउटा वा दुईवटा कार्यान्वयनहरू सुरु देखि अन्त्य सम्म पढ्नुहोस्।

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
यो दस्तावेज़ AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) प्रयोग गरेर अनुवाद गरिएको हो। हामी सही हुन प्रयास गर्छौं, तर कृपया जानकार हुनुस् कि स्वचालित अनुवादमा त्रुटिहरू वा अशुद्धताहरू हुन सक्छन्। मूल दस्तावेज़ यसको मूल भाषामा आधिकारिक स्रोत मानिनुपर्छ। महत्वपूर्ण जानकारीका लागि व्यावसायिक मानव अनुवाद सिफारिस गरिन्छ। यस अनुवादको प्रयोगबाट उत्पन्न कुनै पनि गलत बुझाइ वा त्रुटिको लागि हामी जिम्मेवार छैनौं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
